# CrewAI Travel Planner

| Agent | Tools | Purpose |
|---|---|---|
| User Preference Agent | LLM only | Extracts structured travel requirements from the user's request |
| Destination Research Agent | SerperDevTool + WebsiteSearchTool | Researches destinations, attractions, best seasons, entry requirements |
| Budget Planner Agent | Custom `BudgetCalculatorTool` | Calculates itemized trip cost and checks it against the target budget |
| Transport & Stay Agent | SerperDevTool + WebsiteSearchTool | Finds flights/trains/buses and hotel options |
| Itinerary Planner Agent | WebsiteSearchTool | Builds the day-by-day plan from everyone else's output |
| Food Planner Agent | SerperDevTool + WebsiteSearchTool | Finds restaurants matching dietary preferences |
| Weather & Risk Agent | Custom `WeatherRiskTool` (OpenWeatherMap) | Forecast + basic travel-risk flags |
| Review Agent | LLM only | Checks the whole plan for consistency and produces the final document |


In [20]:
%pip install -q crewai>=0.86.0 crewai-tools>=0.17.0 "requests==2.32.4"

In [39]:
%%capture
%pip install -q langchain-google-genai

In [32]:
import os
from getpass import getpass

def set_key(name, required=True):
    if os.getenv(name):
        return
    # For optional keys like OPENAI_API_KEY which might be checked by tools
    # even when not directly used, set a dummy value if not provided.
    if not required and name == "OPENAI_API_KEY":
        os.environ[name] = "dummy-key"
        print(f"Set dummy value for {name}.")
        return

    value = getpass(f"Enter {name} ({'required' if required else 'optional, press Enter to skip'}): ")
    if value:
        os.environ[name] = value

set_key("GEMINI_API_KEY", required=True)
set_key("SERPER_API_KEY", required=False)
set_key("OPENWEATHER_API_KEY", required=False)
set_key("OPENAI_API_KEY", required=False)

for k in ["GEMINI_API_KEY", "SERPER_API_KEY", "OPENWEATHER_API_KEY", "OPENAI_API_KEY"]:
    print(f"{k}: {'set' if os.getenv(k) else 'not set'}")

Set dummy value for OPENAI_API_KEY.
GEMINI_API_KEY: set
SERPER_API_KEY: set
OPENWEATHER_API_KEY: set
OPENAI_API_KEY: set


In [54]:
from crewai import LLM
import os

# Initialize the CrewAI's LLM class directly with the model string and parameters
llm = LLM(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7,
)

In [22]:
import json
from typing import Type

from crewai.tools import BaseTool
from pydantic import BaseModel, Field


class BudgetCalculatorInput(BaseModel):
    num_travelers: int = Field(..., description="Number of people traveling")
    num_days: int = Field(..., description="Number of days/nights of the trip")
    flight_cost_per_person: float = Field(0, description="Round-trip flight/train/bus cost per person")
    hotel_cost_per_night: float = Field(0, description="Hotel cost per night (per room)")
    num_rooms: int = Field(1, description="Number of hotel rooms needed")
    food_cost_per_day_per_person: float = Field(0, description="Estimated food cost per day per person")
    activities_cost_total: float = Field(0, description="Total activities/attractions cost for the whole group")
    local_transport_cost_total: float = Field(0, description="Total local transport cost for the whole trip")
    misc_buffer_percent: float = Field(10, description="Buffer percentage added on top of the subtotal")
    target_budget: float = Field(0, description="Optional target total budget. Pass 0 if not specified.")


class BudgetCalculatorTool(BaseTool):
    name: str = "Trip Budget Calculator"
    description: str = (
        "Calculates a detailed trip budget breakdown and total cost, and compares it "
        "against an optional target budget. Always use this tool instead of doing budget math yourself."
    )
    args_schema: Type[BaseModel] = BudgetCalculatorInput

    def _run(self, num_travelers, num_days, flight_cost_per_person=0, hotel_cost_per_night=0,
              num_rooms=1, food_cost_per_day_per_person=0, activities_cost_total=0,
              local_transport_cost_total=0, misc_buffer_percent=10, target_budget=0) -> str:
        num_nights = max(num_days - 1, 1)
        flights_total = flight_cost_per_person * num_travelers
        hotel_total = hotel_cost_per_night * num_rooms * num_nights
        food_total = food_cost_per_day_per_person * num_travelers * num_days
        subtotal = flights_total + hotel_total + food_total + activities_cost_total + local_transport_cost_total
        buffer_amount = subtotal * (misc_buffer_percent / 100)
        grand_total = subtotal + buffer_amount
        per_person_total = grand_total / num_travelers if num_travelers else grand_total

        result = {
            "breakdown": {
                "flights_or_transport": round(flights_total, 2),
                "hotel_stay": round(hotel_total, 2),
                "food": round(food_total, 2),
                "activities": round(activities_cost_total, 2),
                "local_transport": round(local_transport_cost_total, 2),
                "misc_buffer": round(buffer_amount, 2),
            },
            "grand_total": round(grand_total, 2),
            "per_person_total": round(per_person_total, 2),
            "num_travelers": num_travelers,
            "num_days": num_days,
        }
        if target_budget > 0:
            diff = round(target_budget - grand_total, 2)
            result.update(target_budget=target_budget, status="within_budget" if diff >= 0 else "over_budget",
                           difference_from_target=diff)
        return json.dumps(result, indent=2)

In [23]:
import requests


class WeatherRiskInput(BaseModel):
    city: str = Field(..., description="Destination city name, e.g. 'Bali'")
    country_code: str = Field("", description="Optional ISO country code, e.g. 'IN'")


class WeatherRiskTool(BaseTool):
    name: str = "Weather and Travel Risk Lookup"
    description: str = "Fetches a 5-day forecast for a city and flags basic weather risks (rain, heat, cold, wind)."
    args_schema: Type[BaseModel] = WeatherRiskInput

    def _run(self, city: str, country_code: str = "") -> str:
        key = os.getenv("OPENWEATHER_API_KEY")
        if not key:
            return json.dumps({"error": "OPENWEATHER_API_KEY not set."})

        q = f"{city},{country_code}" if country_code else city
        try:
            geo = requests.get("https://api.openweathermap.org/geo/1.0/direct",
                                params={"q": q, "limit": 1, "appid": key}, timeout=10).json()
            if not geo:
                return json.dumps({"error": f"Could not find location '{q}'"})
            lat, lon = geo[0]["lat"], geo[0]["lon"]
            data = requests.get("https://api.openweathermap.org/data/2.5/forecast",
                                 params={"lat": lat, "lon": lon, "appid": key, "units": "metric"},
                                 timeout=10).json()
        except requests.RequestException as e:
            return json.dumps({"error": f"Weather API request failed: {e}"})

        days, risks = {}, set()
        for e in data.get("list", []):
            date = e["dt_txt"].split(" ")[0]
            temp, wind, pop = e["main"]["temp"], e["wind"]["speed"], e.get("pop", 0)
            d = days.setdefault(date, {"temps": [], "wind": 0, "pop": 0, "cond": set()})
            d["temps"].append(temp); d["wind"] = max(d["wind"], wind); d["pop"] = max(d["pop"], pop)
            d["cond"].add(e["weather"][0]["main"])
            if pop >= 0.6: risks.add(f"Heavy rain likely on {date}")
            if temp >= 38: risks.add(f"Extreme heat (~{round(temp)}°C) on {date}")
            if temp <= 2: risks.add(f"Extreme cold (~{round(temp)}°C) on {date}")
            if wind >= 15: risks.add(f"High winds (~{round(wind)} m/s) on {date}")

        forecast = [
            {"date": dt, "min_c": round(min(v["temps"]), 1), "max_c": round(max(v["temps"]), 1),
             "conditions": list(v["cond"]), "max_wind_ms": round(v["wind"], 1),
             "rain_probability_pct": round(v["pop"] * 100)}
            for dt, v in sorted(days.items())
        ]
        return json.dumps({
            "location": geo[0].get("name", city),
            "country": geo[0].get("country", country_code),
            "forecast": forecast,
            "risk_flags": sorted(risks) if risks else ["No significant weather risks detected"],
        }, indent=2)

In [24]:
budget_tool = BudgetCalculatorTool()
print(budget_tool._run(num_travelers=2, num_days=5, flight_cost_per_person=25000,
                        hotel_cost_per_night=4000, food_cost_per_day_per_person=800,
                        activities_cost_total=8000, local_transport_cost_total=3000,
                        target_budget=150000))

{
  "breakdown": {
    "flights_or_transport": 50000,
    "hotel_stay": 16000,
    "food": 8000,
    "activities": 8000,
    "local_transport": 3000,
    "misc_buffer": 8500.0
  },
  "grand_total": 93500.0,
  "per_person_total": 46750.0,
  "num_travelers": 2,
  "num_days": 5,
  "target_budget": 150000,
  "status": "within_budget",
  "difference_from_target": 56500.0
}


In [56]:
try:
    from crewai_tools import SerperDevTool, WebsiteSearchTool
    HAS_SEARCH_TOOLS = True
except ImportError:
    HAS_SEARCH_TOOLS = False

search_tool = SerperDevTool() if HAS_SEARCH_TOOLS and os.getenv("SERPER_API_KEY") else None
website_tool = WebsiteSearchTool(llm=llm) if HAS_SEARCH_TOOLS else None
weather_tool = WeatherRiskTool()

def search_tools():
    return [t for t in (search_tool, website_tool) if t]

print("Search tools ready:", [t.name for t in search_tools()] or "none (LLM-only fallback)")

Search tools ready: ['Search the internet with Serper', 'Search in a specific website']


In [57]:
from crewai import Agent

user_preference_agent = Agent(
    role="User Preference Analyst",
    goal=(
        "Extract a clean, structured set of travel requirements from the user's "
        "free-form input (destination or origin+region, dates/duration, number "
        "of travelers, budget, dietary preference, trip style/interests, and any "
        "constraints)."
    ),
    backstory=(
        "You are a meticulous and highly experienced travel consultant who specializes in interviewing "
        "clients and translating vague requests into precise, structured briefs "
        "that the rest of the travel planning team can act on without needing to "
        "ask follow-up questions."
    ),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

destination_research_agent = Agent(
    role="Destination Research Specialist",
    goal=(
        "Research and recommend destinations (or confirm the user's chosen "
        "destination), covering top attractions, best time to visit (if has "
        "given duration or timeline, give best places of that duration), local "
        "culture/etiquette notes, and entry requirements (visa basics, if "
        "international)."
    ),
    backstory=(
        "You are a well-traveled senior-most researcher who has visited hundreds of cities "
        "and always digs up current, accurate, and locally-grounded information "
        "rather than generic guidebook filler."
    ),
    tools=search_tools(),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

budget_planner_agent = Agent(
    role="Budget Planner",
    goal=(
        "Build an accurate, itemized trip budget covering transport, stay, food, "
        "activities, and local transport, including a sensible contingency "
        "buffer, and flag whether it fits the user's target budget."
    ),
    backstory=(
        "You are a former financial analyst turned travel planner who is "
        "obsessive about accurate arithmetic and never eyeballs a total - "
        "you always use your calculator tool for every cost computation."
    ),
    tools=[budget_tool],
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

transport_stay_agent = Agent(
    role="Transport and Accommodation Finder",
    goal=(
        "Find realistic flight/train/bus options and hotel/stay options that "
        "match the traveler's budget, dates, and preferences, with approximate "
        "prices and booking considerations."
    ),
    backstory=(
        "You are one of the best travel agents in the world with two decades of "
        "experience booking transport and stays who knows how to balance cost, "
        "comfort, and convenience for any type of traveler."
    ),
    tools=search_tools(),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

itinerary_planner_agent = Agent(
    role="Itinerary Planner",
    goal=(
        "Combine the destination research, transport/stay details, food "
        "recommendations, and weather/risk information into a coherent "
        "day-by-day itinerary that is realistic about travel time and pacing."
    ),
    backstory=(
        "You are a detail-oriented trip designer known for building itineraries "
        "that people actually enjoy following - never overpacked, always "
        "accounting for rest, travel time between locations, and logical "
        "geographic sequencing."
    ),
    tools=search_tools(),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

food_planner_agent = Agent(
    role="Local Food and Dining Planner",
    goal=(
        "Recommend restaurants and food experiences at the destination that "
        "match the traveler's dietary preference (Veg, Non-Veg / Vegan / Jain, or "
        "other restrictions), spread sensibly across the itinerary."
    ),
    backstory=(
        "You are a food-obsessed local guide who knows how to find great, "
        "dietary-appropriate meals anywhere in the world, from street food to "
        "fine dining and knows everything about restaurants, how they cook, "
        "what they use, if veg and non-veg compartment are seperated or not."
    ),
    tools=search_tools(),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

weather_risk_agent = Agent(
    role="Weather and Travel Risk Analyst",
    goal=(
        "Retrieve the weather forecast for the destination city and flag any "
        "travel risks or advisories relevant to the planned dates. Provide "
        "practical packing suggestions based on the forecast."
    ),
    backstory=(
        "You are a highly cautious travel-safety analyst who always checks the "
        "forecast and current conditions before anyone sets off, and gives "
        "clear, practical packing and safety advice based on what you find."
    ),
    tools=[weather_tool],
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

review_agent = Agent(
    role="Quality Reviewer",
    goal=(
        "Review the complete travel plan for consistency, completeness, and "
        "realism - checking that budget numbers, dates, and locations are "
        "aligned across all sections - and produce a final polished itinerary."
    ),
    backstory=(
        "You are an exacting editor-in-chief for a travel publication who never "
        "lets an inconsistency, an unrealistic timing, or a math error slip "
        "through to the final draft."
    ),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

print("All 8 agents defined with Gemini LLM.")

All 8 agents defined with Gemini LLM.


In [43]:
from crewai import Task

extract_preferences_task = Task(
    description=(
        'Analyze the following user travel request and extract a structured '
        'travel brief:\n\n"{user_request}"\n\n'
        "Extract (and make reasonable assumptions where something is missing, "
        "clearly labeling them as assumptions):\n"
        "- Origin city/country (if mentioned)\n"
        "- Destination (city/country, or \"open to suggestions\" + region/type of "
        "trip if not specified)\n"
        "- Travel dates or duration (number of days)\n"
        "- Number of travelers\n"
        "- Total or per-person budget (and currency)\n"
        "- Dietary preference (Veg / Non-Veg / Vegan / Jain / Other / None specified)\n"
        "- Trip style/interests (e.g. adventure, relaxation, culture, nightlife, "
        "family-friendly, honeymoon, etc.)\n"
        "- Any explicit constraints (mobility issues, must-see places, places to "
        "avoid, pace preference)"
    ),
    expected_output=(
        "A clearly labeled structured brief with each field above, using "
        '"Not specified - assuming X" for any field you had to infer.'
    ),
    agent=user_preference_agent,
)

research_destination_task = Task(
    description=(
        "Using the structured travel brief, research the destination in depth. "
        "If the destination is not fixed, propose 2-3 suitable destinations "
        "matching the trip style and budget, then pick the best one to proceed "
        "with and justify the choice.\n\n"
        "Cover: top 5-8 attractions/experiences, best time of year to visit vs "
        "the planned dates, local culture/etiquette tips, and any visa/entry "
        "requirement basics for international trips."
    ),
    expected_output=(
        "A destination research brief with: chosen destination + justification, "
        "list of top attractions with 1-2 line descriptions, best-time-to-visit "
        "note, culture/etiquette tips, and entry requirement notes."
    ),
    agent=destination_research_agent,
    context=[extract_preferences_task],
)

find_transport_stay_task = Task(
    description=(
        "Based on the travel brief and chosen destination, find realistic "
        "transport options (flights/trains/buses) from the origin to the "
        "destination and back, and 2-3 hotel/stay options at different price "
        "tiers (budget, mid-range, comfort) that fit the traveler's budget and "
        "trip style. Give approximate prices in the budget's currency."
    ),
    expected_output=(
        "A list of 2-3 transport options each way with approximate price and "
        "duration, and 2-3 accommodation options with nightly price, location "
        "notes, and why they suit this traveler."
    ),
    agent=transport_stay_agent,
    context=[extract_preferences_task, research_destination_task],
)

plan_budget_task = Task(
    description=(
        "Using the transport and stay options found, and the group size/trip "
        "length from the brief, calculate a full itemized budget using the "
        "Trip Budget Calculator tool. Pick the most sensible transport and "
        "hotel option (or a couple of scenarios if useful) as inputs. Include "
        "food and activities cost estimates based on the destination research. "
        "Compare the total against the user's stated budget and clearly state "
        "whether the trip fits, and suggest adjustments if it does not."
    ),
    expected_output=(
        "An itemized budget breakdown (transport, stay, food, activities, local "
        "transport, buffer, grand total, per-person total) with a clear "
        "within-budget / over-budget verdict and any recommended adjustments."
    ),
    agent=budget_planner_agent,
    context=[extract_preferences_task, find_transport_stay_task],
)

plan_food_task = Task(
    description=(
        "Based on the destination and the traveler's dietary preference, find "
        "restaurant and food experience recommendations (5-8 options) spanning "
        "different meals/price points, all matching the dietary requirement. "
        "Include at least one iconic local food experience if it fits the "
        "dietary constraint."
    ),
    expected_output=(
        "A list of restaurant/food recommendations with cuisine type, price "
        "range, dietary suitability note, and best meal (breakfast/lunch/"
        "dinner/snack) for each."
    ),
    agent=food_planner_agent,
    context=[extract_preferences_task, research_destination_task],
)

check_weather_risk_task = Task(
    description=(
        "Look up the weather forecast for the destination city and flag any "
        "travel risks or advisories relevant to the planned dates. Provide "
        "practical packing suggestions based on the forecast."
    ),
    expected_output=(
        "A weather summary (temperature range, conditions per day where "
        "available), a list of risk flags (or confirmation there are none), and "
        "a short packing recommendation list."
    ),
    agent=weather_risk_agent,
    context=[extract_preferences_task, research_destination_task],
)

build_itinerary_task = Task(
    description=(
        "Combine all prior research (destination, transport & stay, budget, "
        "food, weather) into a complete day-by-day itinerary covering the full "
        "trip duration. Sequence activities geographically and logically, "
        "include approximate timings, note which meals correspond to which food "
        "recommendations, and factor in the weather/risk notes for pacing "
        "(e.g. indoor activities on a day with a rain risk flag)."
    ),
    expected_output=(
        "A day-by-day itinerary (Day 1, Day 2, ...) with morning/afternoon/"
        "evening activities, meal suggestions tied to the food plan, transport/"
        "check-in/check-out logistics on relevant days, and notes referencing "
        "any weather risks affecting that day."
    ),
    agent=itinerary_planner_agent,
    context=[
        extract_preferences_task,
        research_destination_task,
        find_transport_stay_task,
        plan_food_task,
        check_weather_risk_task,
    ],
)

review_final_plan_task = Task(
    description=(
        "Review the complete travel plan (brief, destination research, transport "
        "& stay, budget, food plan, weather/risk notes, and itinerary) for "
        "consistency and completeness. Check that: dates and trip length match "
        "across all sections, the budget total is consistent with what was "
        "booked, dietary preferences are respected throughout, and the "
        "itinerary accounts for weather risks. Fix any inconsistencies you find "
        "and produce the final, polished, traveler-ready itinerary document."
    ),
    expected_output=(
        "A final, polished, well-formatted travel plan document combining all "
        "sections (Trip Overview, Destination Highlights, Transport & Stay, "
        "Budget Breakdown, Day-by-Day Itinerary with meals, Weather & Packing "
        "Notes), free of inconsistencies, ready to hand to the traveler."
    ),
    agent=review_agent,
    context=[
        extract_preferences_task,
        research_destination_task,
        find_transport_stay_task,
        plan_budget_task,
        plan_food_task,
        check_weather_risk_task,
        build_itinerary_task,
    ],
    output_file="final_itinerary.md",
)

print("All 8 tasks defined.")

All 8 tasks defined.


In [58]:
from crewai import Crew, Process

travel_crew = Crew(
    agents=[user_preference_agent, destination_research_agent, transport_stay_agent, budget_planner_agent,
            food_planner_agent, weather_risk_agent, itinerary_planner_agent, review_agent],
    tasks=[extract_preferences_task, research_destination_task, find_transport_stay_task, plan_budget_task,
           plan_food_task, check_weather_risk_task, build_itinerary_task, review_final_plan_task],
    process=Process.sequential,
    verbose=True,
)

In [61]:
user_request = (
    "Plan a 5-day trip for 2 people from Kanpur to Bali, mid-December, "
    "budget around INR 150,000 total, we're Jain and we dont prefer veg "
    "and non-veg mixed restaurant, and we love beaches and temples, "
    "snorkeling, and a bit of nightlife. Nothing too physically extreme."
)

# Use kickoff_async and await its result to resolve the RuntimeError
result = await travel_crew.kickoff_async(inputs={"user_request": user_request})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0fd1ebfe-f237-4ae0-85cf-0e5fa4ebb365                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following user travel request and extract a structured travel brief:                         │
│                                                                                                                 │
│  "Plan a 5-day trip for 2 people from Kanpur to Bali, mid-December, budget around INR 150,000 total, we're      │
│  Jain and we dont prefer veg and non-veg mixed restaurant, and we love beaches and temples, snorkeling, and a   │
│  bit of nightlife. Nothing too physically extreme."                                                             │
│                                                                                                                 │
│  Extract (and make reasonable assumptions where something is missing, clearly labeling them as assumptions):    │
│  - Origin city/country (if mentioned)                                                                           │
│  - Destination (city/country, or "open to suggestions" + region/type of trip if not specified)                  │
│  - Travel dates or duration (number of days)                                                                    │
│  - Number of travelers                                                                                          │
│  - Total or per-person budget (and currency)                                                                    │
│  - Dietary preference (Veg / Non-Veg / Vegan / Jain / Other / None specified)                                   │
│  - Trip style/interests (e.g. adventure, relaxation, culture, nightlife, family-friendly, honeymoon, etc.)      │
│  - Any explicit constraints (mobility issues, must-see places, places to avoid, pace preference)                │
│  ID: 0d3c9ffb-250f-46f6-ae64-e146824d2c3c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: User Preference Analyst                                                                                 │
│                                                                                                                 │
│  Task: Analyze the following user travel request and extract a structured travel brief:                         │
│                                                                                                                 │
│  "Plan a 5-day trip for 2 people from Kanpur to Bali, mid-December, budget around INR 150,000 total, we're      │
│  Jain and we dont prefer veg and non-veg mixed restaurant, and we love beaches and temples, snorkeling, and a   │
│  bit of nightlife. Nothing too physically extreme."                                                             │
│                                                                                                                 │
│  Extract (and make reasonable assumptions where something is missing, clearly labeling them as assumptions):    │
│  - Origin city/country (if mentioned)                                                                           │
│  - Destination (city/country, or "open to suggestions" + region/type of trip if not specified)                  │
│  - Travel dates or duration (number of days)                                                                    │
│  - Number of travelers                                                                                          │
│  - Total or per-person budget (and currency)                                                                    │
│  - Dietary preference (Veg / Non-Veg / Vegan / Jain / Other / None specified)                                   │
│  - Trip style/interests (e.g. adventure, relaxation, culture, nightlife, family-friendly, honeymoon, etc.)      │
│  - Any explicit constraints (mobility issues, must-see places, places to avoid, pace preference)                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: User Preference Analyst                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Travel Brief:**                                                                                              │
│                                                                                                                 │
│  *   **Origin city/country:** Kanpur, India                                                                     │
│  *   **Destination:** Bali, Indonesia                                                                           │
│  *   **Travel dates or duration:** 5 days, Mid-December (Not specified - assuming current/upcoming year for     │
│  planning purposes)                                                                                             │
│  *   **Number of travelers:** 2                                                                                 │
│  *   **Total or per-person budget:** INR 150,000 total                                                          │
│  *   **Dietary preference:** Jain, prefers pure vegetarian establishments (no mixed veg/non-veg restaurants)    │
│  *   **Trip style/interests:** Beaches, temples (culture/sightseeing), snorkeling (water activities),           │
│  nightlife (moderate)                                                                                           │
│  *   **Any explicit constraints:**                                                                              │
│      *   No physically extreme activities.                                                                      │
│      *   Avoid restaurants that serve both vegetarian and non-vegetarian food.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the complete travel plan (brief, destination research, transport & stay, budget, food plan,       │
│  weather/risk notes, and itinerary) for consistency and completeness. Check that: dates and trip length match   │
│  across all sections, the budget total is consistent with what was booked, dietary preferences are respected    │
│  throughout, and the itinerary accounts for weather risks. Fix any inconsistencies you find and produce the     │
│  final, polished, traveler-ready itinerary document.                                                            │
│  ID: 155bcf33-9a97-47b4-8f7b-fe6c98f77952                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quality Reviewer                                                                                        │
│                                                                                                                 │
│  Task: Review the complete travel plan (brief, destination research, transport & stay, budget, food plan,       │
│  weather/risk notes, and itinerary) for consistency and completeness. Check that: dates and trip length match   │
│  across all sections, the budget total is consistent with what was booked, dietary preferences are respected    │
│  throughout, and the itinerary accounts for weather risks. Fix any inconsistencies you find and produce the     │
│  final, polished, traveler-ready itinerary document.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quality Reviewer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Travel Plan: Bali Bliss for Two (Jain Pure Vegetarian)**                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **1. Trip Overview**                                                                                       │
│                                                                                                                 │
│  *   **Origin City/Country:** Kanpur, India                                                                     │
│  *   **Destination:** Bali, Indonesia                                                                           │
│  *   **Travel Dates:** 5 Days, Mid-December (Exact dates to be confirmed upon booking)                          │
│  *   **Number of Travelers:** 2                                                                                 │
│  *   **Total Budget:** INR 150,000                                                                              │
│  *   **Dietary Preference:** Jain, prefers pure vegetarian establishments (no mixed veg/non-veg restaurants)    │
│  *   **Trip Style/Interests:** Beaches, temples (culture/sightseeing), snorkeling (water activities), moderate  │
│  nightlife                                                                                                      │
│  *   **Constraints:** No physically extreme activities; avoid restaurants that serve both vegetarian and        │
│  non-vegetarian food.                                                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### **2. Destination Highlights: Bali, Indonesia**                                                             │
│                                                                                                                 │
│  Bali, Indonesia, is an ideal destination that perfectly aligns with the travelers' interests. Known as the     │
│  "Island of the Gods," it offers a rich tapestry of cultural experiences, stunning natural beauty, and a        │
│  vibrant atmosphere.                                                                                            │
│                                                                                                                 │
│  **Why Bali?**                                                                                                  │
│  *   **Culture & Temples:** Iconic sea temples like Tanah Lot and Uluwatu, and the spiritual heart of Ubud,     │
│  offer deep cultural immersion.                                                                                 │
│  *   **Beaches & Water Activities:** Seminyak and Kuta provide beautiful coastlines, while nearby Nusa Penida   │
│  is excellent for snorkeling.                          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the complete travel plan (brief, destination research, transport & stay, budget, food plan,       │
│  weather/risk notes, and itinerary) for consistency and completeness. Check that: dates and trip length match   │
│  across all sections, the budget total is consistent with what was booked, dietary preferences are respected    │
│  throughout, and the itinerary accounts for weather risks. Fix any inconsistencies you find and produce the     │
│  final, polished, traveler-ready itinerary document.                                                            │
│  Agent: Quality Reviewer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [62]:
from IPython.display import Markdown, display

display(Markdown(str(result)))

**Travel Plan: Bali Bliss for Two (Jain Pure Vegetarian)**

---

### **1. Trip Overview**

*   **Origin City/Country:** Kanpur, India
*   **Destination:** Bali, Indonesia
*   **Travel Dates:** 5 Days, Mid-December (Exact dates to be confirmed upon booking)
*   **Number of Travelers:** 2
*   **Total Budget:** INR 150,000
*   **Dietary Preference:** Jain, prefers pure vegetarian establishments (no mixed veg/non-veg restaurants)
*   **Trip Style/Interests:** Beaches, temples (culture/sightseeing), snorkeling (water activities), moderate nightlife
*   **Constraints:** No physically extreme activities; avoid restaurants that serve both vegetarian and non-vegetarian food.

---

### **2. Destination Highlights: Bali, Indonesia**

Bali, Indonesia, is an ideal destination that perfectly aligns with the travelers' interests. Known as the "Island of the Gods," it offers a rich tapestry of cultural experiences, stunning natural beauty, and a vibrant atmosphere.

**Why Bali?**
*   **Culture & Temples:** Iconic sea temples like Tanah Lot and Uluwatu, and the spiritual heart of Ubud, offer deep cultural immersion.
*   **Beaches & Water Activities:** Seminyak and Kuta provide beautiful coastlines, while nearby Nusa Penida is excellent for snorkeling.
*   **Moderate Nightlife:** Seminyak offers upscale beach clubs and restaurants for a sophisticated evening.
*   **Jain-Friendly Options:** Bali, particularly Ubud and Seminyak, has a growing number of dedicated pure vegetarian and vegan restaurants, many of which are accustomed to Jain dietary requirements (no onion, no garlic, no root vegetables).

**Top Attractions/Experiences:**
*   **Tanah Lot Temple:** Iconic sea temple known for breathtaking sunsets.
*   **Uluwatu Temple:** Dramatic clifftop temple with ocean views and traditional Kecak fire dance.
*   **Ubud Monkey Forest:** Sacred sanctuary with ancient temples and playful monkeys.
*   **Seminyak Beach:** Golden sands, vibrant beach clubs, and upscale dining.
*   **Tegallalang Rice Terraces (Ubud):** Stunning panoramic views of traditional rice paddies.
*   **Snorkeling in Nusa Penida:** Day trip for vibrant marine life and exploring beautiful coastal viewpoints (e.g., Kelingking Beach viewpoint).
*   **Ubud's Cultural Heart:** Art markets, traditional craft shops, and wellness activities.

**Local Culture/Etiquette Tips:**
*   **Dress Modestly:** Cover shoulders and knees when visiting temples. Sarongs are often required and available for rent.
*   **Respect Offerings:** Be mindful of daily offerings (Canang Sari) on the ground.
*   **Right Hand:** Use your right hand when giving or receiving anything.
*   **Remove Shoes:** When entering homes or temples.
*   **Dietary Communication:** Always clearly state "Jain food," "pure vegetarian," and "no onion, no garlic, no root vegetables" at restaurants.

**Entry Requirements (Indian Citizens):**
*   **Electronic Visa on Arrival (e-VOA):** Required for Indian citizens, apply online before travel.
*   **Passport Validity:** Must be valid for at least six months beyond your departure date from Indonesia.
*   **Bali Tourism Tax:** IDR 150,000 per person, payable online in advance or upon arrival (as of Feb 14, 2024).

---

### **3. Transport & Stay**

**Chosen Transport (Kanpur to Bali and back for 2 travelers):**
*   **Kanpur to Delhi (and back by train):** Vande Bharat Express (AC train)
    *   **Approximate Price:** INR 4,000 (round trip for two)
*   **Delhi (DEL) to Denpasar (DPS), Bali (and back by flight):** Flights with 1 stopover (e.g., via Singapore, Kuala Lumpur). Airlines like Singapore Airlines, Malaysia Airlines, Thai Airways, Batik Air, or IndiGo (with partners).
    *   **Approximate Price:** INR 60,000 (round trip for two)
*   **Total Approximate Transport Cost:** INR 64,000
    *   *Justification:* This cost-effective option balances comfort and budget, utilizing the train for the domestic leg and a connecting international flight. Booking well in advance for mid-December is crucial to secure these prices.

**Chosen Accommodation (4 nights for 2 travelers):**
*   **Suly Vegetarian Resort & Spa, Ubud**
    *   **Nightly Price:** Approximately INR 8,000
    *   **Total for 4 nights:** INR 32,000
    *   **Location Notes:** Located in Ubud, the cultural heart of Bali, amidst tropical greenery. Offers a serene environment.
    *   *Justification:* This resort is explicitly a **fully vegetarian resort**, ensuring all meals served on-site adhere to your pure vegetarian preference without any risk of cross-contamination or mixed menus. Its location is ideal for exploring Ubud's cultural attractions and temples, aligning perfectly with your interests.

---

### **4. Budget Breakdown**

**Trip Budget Breakdown (5 Days, 2 Travelers)**

*   **Target Budget:** INR 150,000

| Category                            | Details                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     | **Flights from Delhi to Denpasar (for 2 people, round trip)** | **60,000**               |
| Accommodation (4 nights)            | Suly Vegetarian Resort & Spa, Ubud (INR 8,000/night x 4 nights) | **32,000**               |
| Food (5 days)                       | INR 1,800/person/day x 2 people x 5 days                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |
| **Flights (Kanpur-Delhi-Denpasar)** | 60,000 (Delhi-Denpasar round trip for 2) + 4,000 (Kanpur-Delhi train round trip for 2) | **64,000**               |
| Accommodation (4 nights)            | Suly Vegetarian Resort & Spa, Ubud (INR 8,000/night x 4 nights)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 | **Estimated Cost (INR)** **(Updated)** |
|-------------------------------------|--------------------------------------|
| **Transport (Flights from Delhi)**  | 4,000 (Kanpur-Delhi train round trip) |
|   *Kanpur-Delhi (Train)*            | 60,000 (Delhi-Denpasar flight round trip for 2) |
| **Accommodation (4 nights)**        | **32,000**                           |
|   *Suly Vegetarian Resort & Spa*    | 8,000/night x 4 nights               |
| **Food**                            | **18,000**                           |
|   *1,800/person/day x 2 people x 5 days* |                                      |
| **Activities**                      | **15,000**                           |
|   *Temples, snorkeling, beach activities, moderate nightlife* |                                      |
| **Local Transport**                 | **6,000**                            |
|   *Taxis/ride-hailing for 5 days*    |                                      |
| **Contingency Buffer (10%)**        | **13,500**                           |
| **Grand Total Estimated Cost**      | **148,500**                          |
| **Per-Person Total**                | **74,250**                           |
| **Target Budget**                   | **150,000**                          |
| **Difference from Target**          | **+1,500** (Under Budget)            |

**Verdict:**
The revised estimated trip cost of **INR 148,500** is now comfortably **within your target budget of INR 150,000**. This budget allows for comfortable travel, dedicated Jain-friendly accommodation and dining, and a good range of activities.

---

### **5. Day-by-Day Itinerary: Bali - Mid-December**

This itinerary combines your interests in beaches, temples, snorkeling, and moderate nightlife with your Jain dietary preferences and considers the mid-December wet season weather. It focuses on a base in Ubud for cultural immersion and easier access to Jain-friendly dining, with well-planned day trips to coastal areas.

**Accommodation:** Suly Vegetarian Resort & Spa, Ubud

---

**Day 1: Arrival in Bali & Ubud Exploration**
*   **Weather Note:** Mid-December wet season typically brings warm, humid weather with a moderate to high chance of afternoon or evening rain showers.
*   **Morning (Arrival & Check-in):**
    *   Arrive at Denpasar Ngurah Rai International Airport (DPS) from Delhi.
    *   Complete e-VOA and immigration formalities. Pay Bali Tourism Tax (IDR 150,000 per person) if not done online.
    *   Meet your pre-arranged private transfer to Ubud (approx. 1.5 - 2 hours drive).
    *   Check into **Suly Vegetarian Resort & Spa**. Settle in and refresh.
*   **Lunch (1:00 PM - 2:00 PM):**
    *   **Meal Recommendation:** Enjoy a comforting pure vegetarian Indian lunch with Jain options at **Sattvik by Nature (Ubud)**. Clearly communicate "Jain food, no onion, no garlic, no root vegetables."
*   **Afternoon (2:30 PM - 5:30 PM):**
    *   **Activity:** Visit the **Ubud Monkey Forest (Sacred Monkey Forest Sanctuary)**. The dense tree canopy offers some natural cover from potential showers. Walk through the lush forest, observe the playful monkeys, and explore the ancient temples within.
    *   *Cultural Tip:* Be mindful of your belongings and avoid direct eye contact with the monkeys.
*   **Evening (6:00 PM onwards):**
    *   **Activity:** Explore Ubud's main street (Jalan Raya Ubud). Browse local shops for souvenirs and traditional crafts.
    *   **Dinner (7:30 PM):**
        *   **Meal Recommendation:** Experience a diverse range of vegan Indonesian dishes at **Sawobali Vegan Buffet (Ubud)**. Confirm allium-free options upon arrival.
    *   **Relax** at your resort.

---

**Day 2: Cultural Immersion & Rice Terraces**
*   **Weather Note:** Expect warm, humid conditions with a moderate to high chance of rain, possibly heavy, in the afternoon or evening.
*   **Morning (9:00 AM - 1:00 PM):**
    *   **Activity:** Start with a visit to **Puri Saren Agung (Ubud Palace)** to admire traditional Balinese architecture. Afterwards, explore the **Ubud Art Market** for souvenirs and local handicrafts. Much of these areas offer some shelter.
    *   *Alternative/Backup Activity if heavy rain:* Consider a pre-booked Balinese cooking class that explicitly caters to Jain dietary needs (essential to confirm this when booking).
*   **Lunch (1:30 PM - 2:30 PM):**
    *   **Meal Recommendation:** Enjoy a healthy and refreshing vegan international meal at **Plant Bistro (Ubud)**. Request allium-free preparation.
*   **Afternoon (3:00 PM - 6:00 PM):**
    *   **Activity:** Visit the iconic **Tegallalang Rice Terraces**. While it might rain, the lush green landscape is still stunning. Enjoy a scenic walk along the terraces, perhaps stopping for a warm drink at a cafe overlooking the paddies.
    *   *Weather Note:* Carry rain gear (umbrella/rain jacket) and wear comfortable, non-slip footwear.
*   **Evening (7:00 PM onwards):**
    *   **Activity:** Indulge in a relaxing Balinese massage at a local spa in Ubud.
    *   **Dinner (8:00 PM):**
        *   **Meal Recommendation:** Savor a unique raw vegan and plant-based dinner at **Sage Bali (Ubud)**, which is known for its Jain-friendly preparations (no onion, garlic, or root vegetables).
    *   **Overnight** in Ubud.

---

**Day 3: Temples & Sunset Views**
*   **Weather Note:** Expect warm, humid conditions with a low to moderate chance of rain, mostly localized showers.
*   **Morning (8:30 AM - 1:00 PM):**
    *   **Activity:** Depart early from Ubud for a day trip to South Bali (approx. 1.5 hours drive).
    *   Visit **Tanah Lot Temple**, an iconic sea temple perched on a rock formation. Explore the temple and its surroundings.
    *   *Cultural Tip:* Dress modestly (shoulders and knees covered) when visiting temples.
*   **Lunch (1:30 PM - 2:30 PM):**
    *   **Meal Recommendation:** Enjoy a casual, allium-free plant-based meal at **The Sloth Bali (Canggu/Pererenan)**, located en route to Uluwatu (approx. 30-45 mins from Tanah Lot).
*   **Afternoon (3:00 PM - 6:30 PM):**
    *   **Activity:** Continue to **Uluwatu Temple** (approx. 1 hour drive from Canggu). Explore the stunning clifftop temple with dramatic ocean views.
    *   **Optional Activity (6:00 PM - 7:00 PM):** Witness the traditional **Kecak Fire Dance** performance at Uluwatu Temple (book tickets in advance).
*   **Evening (7:30 PM onwards):**
    *   **Travel:** Drive back to Ubud (approx. 1.5 - 2 hours).
    *   **Dinner (9:00 PM):**
        *   **Meal Recommendation:** Another opportunity for reliable Jain-friendly Indian food at **Sattvik by Nature (Ubud)** after a long day of sightseeing.
    *   **Overnight** in Ubud.

---

**Day 4: Beach, Snorkeling & Moderate Nightlife**
*   **Weather Note:** Generally sunny with a low chance of rain. Ideal for water activities.
*   **Morning (7:30 AM - 4:00 PM):**
    *   **Activity:** Full-day trip to **Nusa Penida for Snorkeling/Water Activities**.
        *   Early morning transfer from Ubud to Sanur port (approx. 45 mins - 1 hour).
        *   Take a fast boat to Nusa Penida (approx. 30-45 minutes).
        *   Enjoy snorkeling at popular spots like Crystal Bay, Manta Point (seasonal, weather dependent), or Gamat Bay.
        *   Explore the **Kelingking Beach viewpoint** (view from above, as descent to the beach is physically demanding and not recommended).
    *   **Lunch:** Most Nusa Penida tour packages include lunch; ensure you communicate your Jain dietary needs clearly beforehand or pack your own snacks.
    *   **Return:** Fast boat back to Sanur, then transfer back to Ubud.
    *   *Pacing Note:* This is a long day with significant travel. Ensure adequate rest and stay hydrated.
*   **Evening (5:00 PM onwards):**
    *   **Activity:** After returning to Ubud, relax and freshen up.
    *   **Travel (Optional):** If you desire a taste of moderate nightlife, consider a transfer to Seminyak (approx. 1.5 hours) for the evening.
    *   **Activity (Seminyak):** Head to **Seminyak Beach** for sunset views. Enjoy the vibrant atmosphere at one of the beach clubs (e.g., Potato Head or Ku De Ta for drinks, as pure veg Jain options might be limited for dinner).
    *   **Dinner (8:00 PM):**
        *   **Meal Recommendation (Seminyak):** Look for dedicated pure vegetarian Indian restaurants in Seminyak like "Namaskar Thali" or "Queens of Tandoor" for Jain-friendly options.
    *   **Return:** Transfer back to Ubud late evening (approx. 1.5 hours).
    *   **Overnight** in Ubud.
    *   *Alternative Evening (Ubud):* If preferring to stay in Ubud, enjoy a quiet dinner at a local pure vegetarian café and explore more local shops or boutiques.

---

**Day 5: Leisure, Last-Minute Shopping & Departure**
*   **Weather Note:** Generally sunny with a low chance of rain.
*   **Morning (9:00 AM - 12:00 PM):**
    *   **Activity:** Enjoy a leisurely morning.
    *   **Optional Activity:** Visit a local yoga studio for a gentle yoga session (Ubud is renowned for its wellness scene).
    *   **Last-minute Shopping:** Pick up any remaining souvenirs at Ubud Art Market or local boutiques.
*   **Lunch (12:30 PM - 1:30 PM):**
    *   **Meal Recommendation:** Enjoy a final delicious and Jain-friendly meal at **Plant Bistro (Ubud)** or **Sawobali Vegan Buffet (Ubud)**.
*   **Afternoon (2:00 PM onwards):**
    *   **Check-out:** Check out from Suly Vegetarian Resort & Spa.
    *   **Travel:** Private transfer to Denpasar Ngurah Rai International Airport (DPS) (approx. 1.5 - 2 hours).
    *   **Departure:** Depart from Bali to Delhi, then onward to Kanpur.

---

### **6. Weather & Packing Notes**

**Weather in Mid-December (Wet Season):**
Mid-December falls within Bali's wet season (October to April). While the provided detailed forecast was for mid-July 2026, the general conditions described are typical of Bali's wet season.
*   **Temperature:** Expect warm and humid weather throughout, with temperatures generally ranging from 25°C to 32°C.
*   **Rainfall:** Anticipate regular afternoon or evening rain showers. These can sometimes be heavy but typically do not last all day. There will still be plenty of sunshine.
*   **Landscape:** The island is lush and green during this time due to the rain.
*   **Crowds:** Generally less crowded than the peak dry season, though popular areas will still see tourists due to the holiday period.

**Packing Recommendations:**
Given the warm, humid, and potentially rainy conditions, pack smartly:
*   **Lightweight, Breathable Clothing:** Cotton or linen fabrics are ideal to stay comfortable.
*   **Rain Gear:** A lightweight, waterproof jacket or a compact umbrella is essential.
*   **Comfortable Footwear:** Waterproof sandals, flip-flops, or comfortable walking shoes that can handle wet conditions are advisable. Consider an extra pair of shoes.
*   **Swimwear:** For beaches and snorkeling.
*   **Sun Protection:** Sunscreen (even on cloudy days), a wide-brimmed hat, and sunglasses are still important.
*   **Insect Repellent:** Crucial during the wet season.
*   **Modest Attire:** Pack sarongs or light scarves to cover shoulders and knees when visiting temples.
*   **Reusable Water Bottle:** To stay hydrated and reduce plastic waste.
*   **Small Backpack/Daypack:** For carrying essentials during day trips, preferably water-resistant.
*   **Basic First-Aid Kit:** Include any personal medications.

**General Travel Tips:**
*   **Book in Advance:** Especially for flights and popular tours during the holiday season.
*   **Local Transport:** For convenience, pre-book private transfers for longer distances. Ride-hailing apps like Grab and Gojek are useful for shorter trips.
*   **Stay Hydrated:** Drink plenty of water.
*   **Be Flexible:** The weather can be unpredictable; be prepared to adjust your plans slightly if heavy rain occurs.

In [63]:
%pip install -q fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 23.2 MB/s eta 0:00:00


In [67]:
import re
from fpdf import FPDF

def markdown_to_pdf(markdown_text, output_path="final_itinerary.pdf"):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    DEFAULT_FONT_SIZE = 12
    DEFAULT_LINE_HEIGHT = 8 # Height for a single line of text

    pdf.set_font("Helvetica", size=DEFAULT_FONT_SIZE)

    def reset_x():
        pdf.set_x(pdf.l_margin)

    # This function will now render a full markdown line, handling bold parts, and manage its own line break
    def render_markdown_line(line_text, font_size=DEFAULT_FONT_SIZE, is_list_item=False):
        parts = re.split(r"(\\*\\*.*?\\*\\*)", line_text)

        # If it's a list item, add bullet and indent
        if is_list_item:
            pdf.set_x(pdf.l_margin + 10) # Indent
            pdf.set_font("Helvetica", "", font_size)
            pdf.write(DEFAULT_LINE_HEIGHT, "* ") # Changed from '• ' to '* '
        else:
            reset_x()

        for part in parts:
            if not part:
                continue

            if part.startswith("**") and part.endswith("**"):
                pdf.set_font("Helvetica", "B", font_size)
                pdf.write(DEFAULT_LINE_HEIGHT, part[2:-2])
            else:
                pdf.set_font("Helvetica", "", font_size)
                pdf.write(DEFAULT_LINE_HEIGHT, part)

        pdf.ln(DEFAULT_LINE_HEIGHT) # Always add a new line after rendering the full markdown line
        reset_x()

    def write_heading(text, size):
        reset_x()
        pdf.set_font("Helvetica", "B", size)
        # multi_cell automatically handles wrapping and moves to next line
        pdf.multi_cell(0, DEFAULT_LINE_HEIGHT * 1.2, text, align="L")
        pdf.ln(DEFAULT_LINE_HEIGHT * 0.5) # Add a consistent small space after headings
        reset_x()

    for raw_line in markdown_text.split("\n"):
        line = raw_line.rstrip() # Keep original whitespace if it affects markdown parsing like lists

        if not line:
            pdf.ln(DEFAULT_LINE_HEIGHT * 0.5) # Smaller line break for blank lines
            continue

        if line.startswith("### "):
            pdf.ln(DEFAULT_LINE_HEIGHT * 0.8) # Space before H3
            write_heading(line[4:], 13)
        elif line.startswith("## "):
            pdf.ln(DEFAULT_LINE_HEIGHT * 1.2) # Space before H2
            write_heading(line[3:], 15)
        elif line.startswith("# "):
            pdf.ln(DEFAULT_LINE_HEIGHT * 1.5) # More space before H1
            write_heading(line[2:], 18)
        elif line.startswith(('- ', '* ')): # Corrected to handle both - and * for lists
            pdf.ln(DEFAULT_LINE_HEIGHT * 0.2) # Small space between list items
            render_markdown_line(line[2:], font_size=DEFAULT_FONT_SIZE, is_list_item=True)
        else:
            render_markdown_line(line, font_size=DEFAULT_FONT_SIZE) # Regular paragraph

    pdf.output(output_path)
    return output_path

pdf_path = markdown_to_pdf(str(result), "final_itinerary.pdf")
print(f"PDF saved to: {pdf_path}")

PDF saved to: final_itinerary.pdf
